# G2: Rotations, coaching levers and the language model

Game days 2 and 3, from the inside. G1 showed how the engine plays one chance; this notebook shows how it became a game you can **coach**:

1. **A game you can pause**: `Game.step()`, the loop a live app needs.
2. **Fatigue**: one number per player that drives the rotation.
3. **Calibration**: measure real games first, then tune the simulator's knobs against them (the same idea as picking λ for RAPM).
4. **Levers**: how far a coach's words can push a team, with limits measured from real teams.
5. **Common random numbers**: measuring a small effect without drowning in luck.
6. **The translator**: words → JSON from a language model → a validator → the engine.
7. **Overfitting without a neural network**: why a score on data you tuned to means nothing.

Run the tour cells and read the output. The **✍️ YOUR CODE** cells are exercises; the **✅ CHECK** cell after each one tells you if you got it. Needs the 2025-26 season downloaded and a fitted model (`uv run hoopformer actions`). Use the **Hoopformer (.venv)** kernel.

In [ ]:
# Setup: run this first. It moves to the project folder so paths like data/... work,
# and reloads project code automatically whenever src/hoopformer/ changes.
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("working in", Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
from hoopformer.game import coach, levers, translator
from hoopformer.game.engine import Game, Athlete, stint_seconds, REST_SECONDS, TIRED
from hoopformer.game.levers import lever_limits, validate
from hoopformer.game.model import ActionModel

DATA = Path("data")
OKC, BOS, SGA, TATUM = 1610612760, 1610612738, 1628983, 1628369
model = ActionModel.load(DATA / "derived" / "action_model_2025-26.json")
limits = lever_limits(DATA, "2025-26")   # measured once from box scores, then cached
print(len(model.players), "players |", "home make factor", round(model.league.home_make_factor, 4))

## 1. A game you can pause

Day 1's engine had one loop: `play()` ran the whole game. A coach needs to step in **between possessions**, so the loop became `step()`: play one possession, let both coaches decide timeouts and substitutions, return what happened. `play()` is now just `while not game.done: game.step()`.

In [ ]:
game = Game(model, OKC, BOS, seed=7)
for event in game.step():
    print(f"Q{event.period} {int(event.clock) // 60}:{int(event.clock) % 60:02d}  {event.kind:12s} {event.text}")
print("score", game.away.points, "-", game.home.points, "| clock", round(game.clock, 1), "| game over:", game.done)

### ✍️ Exercise 1: step through the first quarter
With a **new** game (`Game(model, OKC, BOS, seed=7)`), call `step()` until the period changes to 2. Collect every event into a list `q1`, count the steps in `steps`, and count the events of kind `"sub"` and `"timeout"` into `subs` and `timeouts_q1`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 1
full = Game(model, OKC, BOS, seed=7).play()          # the same seed, played straight through
expected = [e for e in full.events if e.period == 1]
print(f"{steps} steps, {subs} substitutions, {timeouts_q1} timeouts in the first quarter")
assert [e.text for e in q1] == [e.text for e in expected], "stepping and play() must give the same game"
assert subs == sum(e.kind == "sub" for e in expected) and timeouts_q1 == sum(e.kind == "timeout" for e in expected)
assert steps > 40, "about one step per possession"
print("same seed, same game, whether you step or play: that's what makes a live game replayable")

## 2. Fatigue: one number per player

Each athlete has an **energy** from 1 (fresh) to 0. On the floor it drains so that it reaches `TIRED` (0.25) after `stint_seconds(minutes per game)`; on the bench it recovers at a fixed rate (`REST_SECONDS` = 240 s from 0.25 back to 1).

Where did `stint_seconds` come from? From the data: real 2025-26 stints last about 5.4 minutes for players under 20 minutes a game and 9 minutes for players over 33. A straight line through those points: **2.5 + 0.17 × minutes per game**, kept between 6 and 10 minutes.

A finding worth remembering: real shooting does **not** drop late in a stint (made/expected 0.98–1.01 at every stint length). Coaches sub players out before fatigue shows. So in the engine, fatigue decides **who plays**, not how well they shoot.

In [ ]:
for mpg in (12, 24, 36):
    print(f"{mpg} min/game: tired after {stint_seconds(mpg) / 60:.1f} minutes on the floor")

star = Athlete(model.players[SGA])
timeline = []
for minute in range(14):
    on_floor = minute < 9            # 9 minutes on, then 5 on the bench
    star.run(60, on_floor)
    timeline.append({"minute": minute + 1, "on floor": on_floor, "energy": round(star.energy, 3)})
pd.DataFrame(timeline).set_index("minute").T

### ✍️ Exercise 2: energy by hand
Write `energy_after(minutes_per_game, on_seconds, off_seconds)`: start at 1.0, play `on_seconds` (drain `(1 - TIRED) / stint_seconds(minutes_per_game)` per second, never below 0), then rest `off_seconds` (recover `(1 - TIRED) / REST_SECONDS` per second, never above 1). Return the energy.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 2
from dataclasses import replace
for mpg, on, off in ((36, 540, 0), (36, 540, 120), (14, 900, 60), (30, 200, 400)):
    athlete = Athlete(replace(model.players[SGA], minutes_per_game=mpg))
    athlete.run(on, True)
    athlete.run(off, False)
    print(f"{mpg} mpg, {on}s on, {off}s off -> yours {energy_after(mpg, on, off):.4f}, engine {athlete.energy:.4f}")
    assert abs(energy_after(mpg, on, off) - athlete.energy) < 1e-9

## 3. Calibration: measure first, then tune

The coach has knobs nobody can read off a box score: how far ahead of their minutes a player may get, how far behind a rested player must be before they come back in. So I measured **real** rotations from the rebuilt lineups, then tuned the knobs until simulated games looked the same.

This is the same move as choosing λ for RAPM or C for baseline B2: a few candidate settings, a measure of error on data, keep the best. Only here the "model" is a simulator, and the error compares statistics instead of predictions.

In [ ]:
from hoopformer.game.realism import real_rotations, simulate_league, simulated_rotations

real = real_rotations(DATA, "2025-26", model)              # ~6 s: every 2025-26 game
sim = simulated_rotations(simulate_league(model, 300, seed=1), model)
rows = [{"measure": key, "real": real[key], "simulated": sim[key]} for key in ("subs per team-game", "timeouts per team-game")]
rows += [{"measure": f"stint, {tier}", "real": minutes, "simulated": sim["stint minutes"][tier]} for tier, minutes in real["stint minutes"].items()]
pd.DataFrame(rows).round(2)

### ✍️ Exercise 3: pick a knob by grid search
`coach.SWAP_SECONDS` (now 240) is how far behind their share of minutes a rested player must be before they replace the teammate furthest ahead. The cell below simulates three candidates. Write `cost(stats)` that is **0 when `stats` equals `real`** and grows with the error. Use the substitutions and the four stint lengths: for example the relative error in subs plus the average relative error in stints. Then store the candidate with the lowest cost in `best`.

In [ ]:
candidates = {}
for swap in (120.0, 240.0, 400.0):
    coach.SWAP_SECONDS = swap
    candidates[swap] = simulated_rotations(simulate_league(model, 150, seed=2), model)
    print(f"SWAP_SECONDS={swap:.0f}: subs {candidates[swap]['subs per team-game']:.1f},",
          "stints", [round(m, 1) for m in candidates[swap]["stint minutes"].values()])
coach.SWAP_SECONDS = 240.0   # put the real setting back

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 3
print({swap: round(cost(stats), 3) for swap, stats in candidates.items()}, "-> best:", best)
assert cost(real) == 0, "no error when the simulation equals the real games"
assert best == min(candidates, key=lambda swap: cost(candidates[swap]))
assert best != 120.0, "120 s churns players in and out: about 40 substitutions a game, real teams make 26"

I used a bigger version of this (three knobs, 400 games each, minutes accuracy in the cost too). One trade-off it exposed: the setting that matches real stints best leaves stars ~2 minutes short of their real minutes. I chose a setting with slightly too many substitutions (28 vs 26) but stars within 0.7 minutes. **A cost function is a decision about what matters**, not just arithmetic.

## 4. Levers: how far can words push a team?

A coach's words become lever values from -1 to 1. The engine turns a value into a **multiplier** on its rates. How far is +1? I measured it: across the 30 real teams, the highest and lowest rate relative to the league average. "+1 threes" makes a team shoot threes like the season's most three-happy team, and no further.

In [ ]:
pd.DataFrame(limits, index=["lowest real ratio", "highest real ratio"]).T.round(3)

Read it: real teams' **pace** differs by only ±4%, so "play fast" can't do much. Offensive rebounding varies by -19% to +34%, so "crash the glass" can do a lot.

### ✍️ Exercise 4: the multiplier
Write `my_multiplier(value, low, high)`: clamp `value` to [-1, 1]; 0 gives 1; +1 gives `high`; -1 gives `low`; in between, straight lines (positive values interpolate between 1 and `high`, negative ones between 1 and `low`).

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 4
for value in (-3, -1, -0.5, 0, 0.25, 0.6, 1, 2):
    mine, engine = my_multiplier(value, *limits["crash_glass"]), levers.multiplier(value, limits["crash_glass"])
    print(f"crash_glass {value:+}: yours {mine:.4f}, engine {engine:.4f}")
    assert abs(mine - engine) < 1e-12

### Why "+1 threes" doesn't multiply threes by 1.196
The engine scales the **three** weight and then renormalizes the player's three shot zones, so the number of shots stays the same and twos make room. If a player's weights are `rim`, `mid`, `three` and the multiplier is `m`, their share of threes goes from `three / (rim + mid + three)` to `three·m / (rim + mid + three·m)`.

### ✍️ Exercise 5
Using `model.players[SGA].events`, compute SGA's share of shots that are threes before (`share_before`) and after (`share_after`) a three_point_rate lever of +1 (`m = levers.multiplier(1, limits["three_point_rate"])`).

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 5
game = Game(model, OKC, BOS, seed=1, limits=limits)
game.instruct(game.home, validate({"team": {"three_point_rate": 1}}, "more threes", set(game.home.athletes), set(game.away.athletes), None, "notebook"))
m = levers.multiplier(1, limits["three_point_rate"])
weights = game._player_weights(game.home, game.away, SGA, {"rim": 1.0, "mid": 1.0, "three": m}, 1.0, 1.0)
engine_share = weights["three"] / (weights["rim"] + weights["mid"] + weights["three"])
print(f"SGA threes: {share_before:.1%} of shots -> {share_after:.1%} (x{share_after / share_before:.3f}, while m = {m:.3f})")
assert abs(share_after - engine_share) < 1e-12 and share_after / share_before < m

## 5. Common random numbers

How much does "crash the glass" add? Games are noisy: OKC's offensive rebounds swing by ±4 from game to game, while the lever moves the average by about 3. Compare 200 games with the lever against 200 **other** games without it and luck blurs the answer.

The trick from G1: play **the same seeds** both ways. Each pair of games starts from the same dice, so the difference within a pair is mostly the lever. Let's measure how much sharper that is.

In [ ]:
def okc_orebs(raw, seeds):
    """OKC offensive rebounds per game over these seeds, with an instruction at tip-off (or none)."""
    totals = []
    for seed in seeds:
        game = Game(model, OKC, BOS, seed=seed, limits=limits)
        if raw:
            game.instruct(game.home, validate(raw, "notebook", set(game.home.athletes), set(game.away.athletes), None, "notebook"))
        result = game.play()
        totals.append(sum(a.stats["OREB"] for a in result.home.athletes.values()))
    return np.array(totals)

crash = {"team": {"crash_glass": 1.0}}
without = okc_orebs(None, range(200))
with_lever_same_seeds = okc_orebs(crash, range(200))
with_lever_other_seeds = okc_orebs(crash, range(1000, 1200))
print("average OREB without the lever:", without.mean().round(2), "| with:", with_lever_same_seeds.mean().round(2))

### ✍️ Exercise 6: the standard error of the effect, two ways
1. **Paired** (same seeds): the per-game differences `with_lever_same_seeds - without`; their standard error is `std(differences, ddof=1) / sqrt(n)`. Store it in `se_paired`.
2. **Independent** (other seeds): the effect is `with_lever_other_seeds.mean() - without.mean()`; its standard error is `sqrt(var(a, ddof=1)/n + var(b, ddof=1)/n)`. Store it in `se_independent`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 6
effect = (with_lever_same_seeds - without).mean()
print(f"crash the glass: +{effect:.2f} offensive rebounds a game; standard error {se_paired:.3f} paired vs {se_independent:.3f} independent")
print(f"to be as sure without common random numbers you'd need about {(se_independent / se_paired) ** 2:.1f}x as many games")
assert abs(se_independent - np.sqrt(with_lever_other_seeds.var(ddof=1) / 200 + without.var(ddof=1) / 200)) < 1e-9
assert se_paired < se_independent

The same reasoning will matter for your transformer: compare two models on **the same** validation possessions, never on different samples.

### No free wins
Crashing the glass is worth about +3 points a game in the engine, but real teams don't crash every time. The cost is transition defense. So every lever got a side effect: crashing concedes better fast breaks, pressing concedes layups and fouls, protecting the ball attacks the rim less. Their sizes are **assumptions**, named constants in `levers.py`, until your transformer can measure them.

## 6. The translator: words → JSON → validator → engine

The language model never touches the engine. It gets a **system prompt** (the lever menu and the rules) and a **context** (score, clock, both rosters with ids, who the coach is talking to), and must answer with one JSON object. Then `validate` clamps every value, checks every player id against the rosters and puts anything else in `unmapped`. Only then does the engine see it.

In [ ]:
print(translator.SYSTEM[:1500], "...")

In [ ]:
game = Game(model, OKC, BOS, seed=1, limits=limits)
for _ in range(60):
    game.step()
context = translator.context_for(game, game.home, addressed=SGA)
print({key: context[key] for key in ("score", "period", "clock", "addressed_to")})
print("your players:", [(p["person_id"], p["name"]) for p in context["your_players"]][:5], "...")

What a model might send back, and what the validator makes of it (a made-up answer with three mistakes in it):

In [ ]:
raw = {"team": {"pace": 3, "zone_defense": 1}, "players": [{"person_id": TATUM, "aggression": 1}],
       "reply": "Pushing it, coach.", "replier": SGA}
instruction = validate(raw, "Push the pace!", set(game.home.athletes), set(game.away.athletes), None, "made up")
names = {pid: a.profile.name for side in (game.home, game.away) for pid, a in side.athletes.items()}
print("levers:", instruction.describe(names))
print("pace was clamped from 3 to", instruction.team["pace"], "| Tatum isn't on OKC, zone_defense isn't a lever")

### ✍️ Exercise 7: be the language model
The coach says **"Pack the paint and double Tatum."** Write the `raw` JSON an LLM should return (as a Python dict, using the keys in the system prompt above), validate it into `instruction`, and apply it with `game.instruct(game.home, instruction)`.

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 7
print(instruction.describe(names))
assert instruction.team.get("protect_paint", 0) > 0 and instruction.double_team == TATUM
assert game.home.tactics.get("protect_paint", 0) > 0 and game.home.double_team == TATUM, "the engine now has it in memory"
print("Directives in OKC's memory:", [(d.lever, d.value, d.words) for d in game.home.directives])

**Optional, costs about 1,600 tokens:** if `DASHSCOPE_API_KEY` is in `.env`, ask Qwen for real. Change the words and run it a few times.

In [ ]:
translator.load_env()
if os.environ.get("DASHSCOPE_API_KEY"):
    live = translator.translate("Chet, you've got four fouls. Stay down on their pump fakes.", game, game.home,
                                addressed=1631096, use="qwen")
    print(live.source, "->", live.describe(names))
    print("reply:", live.reply)
else:
    print("no DASHSCOPE_API_KEY: skipping the live call")

## 7. Overfitting without a neural network

The keyword-rules translator (`translator.RULES`) is a list of regular expressions I wrote **while looking at** the 50 test phrases. Watch its score on phrases it has never seen.

In [ ]:
sets = {"50 test phrases (rules written for them)": translator.PHRASES,
        "30 held out": translator.HOLDOUT_PHRASES, "20 held out, second set": translator.HOLDOUT2_PHRASES}
quiet = lambda line: None
for label, path in sets.items():
    passed, total, _ = translator.check_phrases(game, game.home, "rules", path, log=quiet)
    print(f"rules on {label}: {passed}/{total}")

For comparison, Qwen (`qwen3.8-max`) on the same sets, 2026-10-02:

| Phrase set | Rules | Qwen, first prompt | Qwen, fixed prompt |
|---|---|---|---|
| 50 test phrases | 50/50 | 47/50 | 50/50 |
| 30 held out | 5/30 | 28/30 | 30/30 |
| 20 held out, written before the fix | 5/20 | not run | **19/20** |

The first prompt's misses were one pattern: told something in one player's talk row, Qwen changed the whole team. After I read the misses on the 30 held-out phrases and fixed the prompt, those 30 were no longer "held out": the prompt had learned from them through me. Only the fresh set of 20 still measured honestly. That's why your test split is locked behind `--final`.

### ✍️ Exercise 8: feel it yourself
Some second-set phrases fail with the rules, e.g. "Hit the offensive glass hard." and "Let's run, run, run.". Add **one** rule to `translator.RULES` (a tuple `(regex, {"team.crash_glass": 0.6})`, or `"team.pace"`, etc.) that makes at least one of them pass. Then compute the second-set score again and store it in `after`.

In [ ]:
before, total2, _ = translator.check_phrases(game, game.home, "rules", translator.HOLDOUT2_PHRASES, log=quiet)
print("second set before your rule:", before, "/", total2)

In [ ]:
# ✍️ YOUR CODE

In [ ]:
# ✅ CHECK 8
seen, _, _ = translator.check_phrases(game, game.home, "rules", translator.PHRASES, log=quiet)
print(f"second set: {before}/{total2} -> {after}/{total2}; the original 50 still {seen}/50")
assert after > before, "your rule fixed a phrase"
assert seen == 50, "...without breaking the ones the rules were written for"
translator.RULES.pop()   # take it back out, so the module is as it was
print("Now ask yourself: is", f"{after}/{total2}", "an honest score for the rules any more?")

It isn't: you wrote the rule while looking at those phrases. To measure the rules honestly again you'd need phrases nobody has looked at.

## 8. Memory: what the athletes remember

Every instruction becomes a **directive** with the coach's words, when it started and when it lapses (the rest of the game, the quarter, or N possessions). `Game.memory` gives one athlete's view: energy, fouls, confidence, the directives that concern them and their last 10 actions.

In [ ]:
game = Game(model, OKC, BOS, seed=3, limits=limits)
for _ in range(30):
    game.step()
raw = {"team": {"pace": 0.6}, "players": [{"person_id": SGA, "aggression": 0.7, "confidence": 0.5}], "duration": "quarter",
       "reply": "Say less. Attacking.", "replier": SGA}
game.instruct(game.home, validate(raw, "Push it this quarter. Shai, take over.", set(game.home.athletes), set(game.away.athletes), SGA, "notebook"))
for _ in range(8):
    game.step()
memory = game.memory(game.home, SGA)
print({key: memory[key] for key in ("name", "energy", "fouls", "confidence")})
for d in memory["directives"]:
    print(f"  {d.lever} = {d.value}  ('{d.words}', until {d.until:.0f}s)")
print("recent:", memory["recent"][-4:])

## Done

What carries over to your transformer (v0.2):
- **Calibration ↔ validation.** Tuning a knob against real data is choosing a hyperparameter on a validation split.
- **Common random numbers ↔ paired comparisons.** Compare two models on the same possessions.
- **Held-out phrases ↔ the locked test split.** Once you've looked, it's no longer a test.
- **Validator ↔ never trusting a model's raw output.** The engine only takes what passes `validate`.

Questions to think about for day 4 (the live app):
- The language model takes about 3 seconds. What should the game do while it thinks?
- "Box out" and "switch everything" land in `unmapped`. Which one would you build as the next lever, and how would you measure its limits?